In [1]:
# Verificación e instalación segura de tabulate
try:
    from tabulate import tabulate
    print("✅ El paquete 'tabulate' ya está instalado. Puedes usarlo directamente.")
except ImportError:
    print("⚠️ 'tabulate' no está instalado. Instalando...")
    try:
        import sys
        !{sys.executable} -m pip install tabulate --quiet
        from tabulate import tabulate
        print("✅ 'tabulate' se instaló correctamente.")
    except Exception as e:
        print(f"❌ Error al instalar 'tabulate': {str(e)}")
        print("Recomendación: Usa 'display()' de pandas como alternativa.")



StatementMeta(, a20f22a8-18c0-4f68-a6d1-70e9c1c369bf, 3, Finished, Available, Finished)

⚠️ 'tabulate' no está instalado. Instalando...
✅ 'tabulate' se instaló correctamente.


In [2]:

# Librerías necesarias
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

from sklearn.model_selection import RepeatedKFold, cross_val_score
from sklearn.metrics import make_scorer, mean_absolute_error, mean_squared_error, r2_score

from pyspark.sql import SparkSession
from sklearn.preprocessing import RobustScaler

import numpy as np
import pandas as pd

import time




StatementMeta(, a20f22a8-18c0-4f68-a6d1-70e9c1c369bf, 4, Finished, Available, Finished)

In [3]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
# Configuración inicial


# ─────────────────────────────────────────────────────────────
# Verificación inicial de dataset Gold para modelado (nuevo enfoque)
# ─────────────────────────────────────────────────────────────

# Cargar datos desde la tabla Gold ya preparada
df_spark = spark.read.table("Gold.bd_aranda_gold_modelo")
df = df_spark.toPandas()

# Información general
print("[INFO] Dataset cargado correctamente desde bd_aranda_gold_modelo.")
print(f"[INFO] Dimensión del DataFrame: {df.shape}")

# Visualizar tipos de datos
print("\n[INFO] Tipos de datos por columna:")
print(df.dtypes)

# Verificar si existen columnas tipo 'object' (no deseadas en esta etapa)
columnas_object = df.select_dtypes(include='object').columns.tolist()
columnas_object = [col for col in columnas_object if col != 'Caso']  # Excluir identificador

if not columnas_object:
    print("\n✅ Todas las columnas, excepto 'Caso', están en formato numérico. No se requiere codificación adicional.")
else:
    print("\n⚠️ Se detectaron columnas tipo 'object' que no deberían estar en el dataset codificado:")
    print("   →", columnas_object)
    print("   Recomendación: revisar si estas columnas requieren codificación antes del modelado.")

# Confirmar que 'tiempo_resolucion_dias' está presente
if 'tiempo_resolucion_dias' not in df.columns:
    raise ValueError("❌ La variable objetivo 'tiempo_resolucion_dias' no está presente en el DataFrame.")
else:
    print("\n✅ Variable objetivo 'tiempo_resolucion_dias' encontrada correctamente.")



StatementMeta(, a20f22a8-18c0-4f68-a6d1-70e9c1c369bf, 5, Finished, Available, Finished)

[INFO] Dataset cargado correctamente desde bd_aranda_gold_modelo.
[INFO] Dimensión del DataFrame: (12736, 66)

[INFO] Tipos de datos por columna:
Caso                                       object
tiempo_resolucion_dias                      int64
SLA_Minutos                                 int64
Tipo_de_Caso_Requerimiento                   bool
Criticidad_BAJA                              bool
                                            ...  
Aplicación_SAP___NUEVAS_FUNCIONALIDADES      bool
Aplicación_SAP___SOLICITUD_DE_ACCESO         bool
Aplicación_SAP_CRM                           bool
Aplicación_SAP_ERP                           bool
Aplicación_SAP_ISU                           bool
Length: 66, dtype: object

✅ Todas las columnas, excepto 'Caso', están en formato numérico. No se requiere codificación adicional.

✅ Variable objetivo 'tiempo_resolucion_dias' encontrada correctamente.


In [4]:
# ─────────────────────────────────────────────────────────────
# 5.4.4. Preparación de Datos para Modelado Predictivo - Versión Final (con control de outliers)
# ─────────────────────────────────────────────────────────────

from sklearn.preprocessing import RobustScaler
import pandas as pd
import numpy as np

def preparar_datos(df, verbose=True, return_scaler=True, filtrar_outliers=False):
    """
    Prepara los datos para modelado predictivo en el contexto del TFM.

    Parámetros:
    -----------
    df : pd.DataFrame
        Conjunto de datos cargado desde bd_aranda_gold_modelo
    verbose : bool
        Si True, imprime mensajes de trazabilidad
    return_scaler : bool
        Si True, devuelve el objeto scaler entrenado (para reutilización)
    filtrar_outliers : bool
        Si True, elimina los outliers en la variable objetivo

    Retorna:
    --------
    tuple:
        X_escalado : pd.DataFrame (para modelos lineales)
        X_original : pd.DataFrame (para árboles)
        y          : pd.Series
        prep_metadata : dict
        scaler     : RobustScaler (opcional)
    """

    if verbose:
        print("\n🧠 INICIO DE PREPARACIÓN DE DATOS PARA MODELADO")
        print("─" * 60)

    # 1. Variables
    columnas_modelo = df.columns.tolist()
    columnas_modelo.remove("Caso")
    target = "tiempo_resolucion_dias"
    if target not in columnas_modelo:
        raise ValueError("❌ La variable objetivo 'tiempo_resolucion_dias' no está en el DataFrame.")
    features = [col for col in columnas_modelo if col != target]

    if verbose:
        print(f"✔ Variables predictoras: {len(features)} columnas")
        print(f"✔ Variable objetivo     : {target}")
        print("─" * 60)

    # 2. Separación
    X = df[features].copy()
    y = df[target].copy()

    # 2.5 Eliminación de outliers si aplica
    outliers_detectados = 0
    if filtrar_outliers:
        q1, q3 = y.quantile(0.25), y.quantile(0.75)
        iqr = q3 - q1
        lim_inf, lim_sup = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        mascara = (y >= lim_inf) & (y <= lim_sup)
        outliers_detectados = (~mascara).sum()
        X = X[mascara]
        y = y[mascara]
        if verbose:
            print(f"📉 Outliers eliminados: {outliers_detectados}")
    else:
        if verbose:
            print("📈 Outliers conservados en los datos.")

    # 3. Escalado robusto
    columnas_escalables = ['SLA_Minutos']
    scaler = RobustScaler()
    X_escalado = X.copy()
    X_escalado[columnas_escalables] = scaler.fit_transform(X[columnas_escalables])

    if verbose:
        print(f"📏 Escalado aplicado a: {columnas_escalables}")
        print(f"🧾 Dimensión de X escalado: {X_escalado.shape}")
        print(f"🧾 Dimensión de X original: {X.shape}")
        print("─" * 60)

    # 4. Resumen de outliers (en y)
    if verbose and not filtrar_outliers:
        q1, q3 = y.quantile(0.25), y.quantile(0.75)
        iqr = q3 - q1
        outliers = y[(y < q1 - 1.5 * iqr) | (y > q3 + 1.5 * iqr)]
        print(f"📊 Outliers en y: {len(outliers)} ({len(outliers)/len(y)*100:.2f}%)")

    # 5. Metadata
    prep_metadata = {
        'features': features,
        'target': target,
        'scaler_aplicado': columnas_escalables,
        'sample_size': len(y),
        'missing_X': int(X.isnull().sum().sum()),
        'missing_y': int(y.isnull().sum()),
        'outliers_eliminados': int(outliers_detectados),
        'prep_date': pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S")
    }

    print("[✔] Preparación completada correctamente.")
    print("─" * 60)

    return (X_escalado, X, y, prep_metadata, scaler) if return_scaler else (X_escalado, X, y, prep_metadata)



StatementMeta(, a20f22a8-18c0-4f68-a6d1-70e9c1c369bf, 6, Finished, Available, Finished)

In [5]:
# ─────────────────────────────────────────────────────────────
# EJECUCIÓN PRINCIPAL - CON OUTLIERS (versión baseline)
# ─────────────────────────────────────────────────────────────

df_spark = spark.read.table("Gold.bd_aranda_gold_modelo")
df = df_spark.toPandas()

X_ridge_con, X_rf_con, y_con, metadata_con, scaler_con = preparar_datos(df, verbose=True, filtrar_outliers=False)

print("\n📌 Datos CON OUTLIERS listos para modelado:")
print(f"   ▸ X_ridge: {X_ridge_con.shape}")
print(f"   ▸ X_rf_xgb: {X_rf_con.shape}")
print(f"   ▸ y: {y_con.shape}")


StatementMeta(, a20f22a8-18c0-4f68-a6d1-70e9c1c369bf, 7, Finished, Available, Finished)


🧠 INICIO DE PREPARACIÓN DE DATOS PARA MODELADO
────────────────────────────────────────────────────────────
✔ Variables predictoras: 64 columnas
✔ Variable objetivo     : tiempo_resolucion_dias
────────────────────────────────────────────────────────────
📈 Outliers conservados en los datos.
📏 Escalado aplicado a: ['SLA_Minutos']
🧾 Dimensión de X escalado: (12736, 64)
🧾 Dimensión de X original: (12736, 64)
────────────────────────────────────────────────────────────
📊 Outliers en y: 1663 (13.06%)
[✔] Preparación completada correctamente.
────────────────────────────────────────────────────────────

📌 Datos CON OUTLIERS listos para modelado:
   ▸ X_ridge: (12736, 64)
   ▸ X_rf_xgb: (12736, 64)
   ▸ y: (12736,)


In [6]:
# ─────────────────────────────────────────────────────────────
# EJECUCIÓN ALTERNATIVA - SIN OUTLIERS (experimento)
# ─────────────────────────────────────────────────────────────

df_spark = spark.read.table("Gold.bd_aranda_gold_modelo")
df = df_spark.toPandas()

X_ridge_sin, X_rf_sin, y_sin, metadata_sin, scaler_sin = preparar_datos(df, verbose=True, filtrar_outliers=True)

print("\n📌 Datos SIN OUTLIERS listos para modelado:")
print(f"   ▸ X_ridge: {X_ridge_sin.shape}")
print(f"   ▸ X_rf_xgb: {X_rf_sin.shape}")
print(f"   ▸ y: {y_sin.shape}")


StatementMeta(, a20f22a8-18c0-4f68-a6d1-70e9c1c369bf, 8, Finished, Available, Finished)


🧠 INICIO DE PREPARACIÓN DE DATOS PARA MODELADO
────────────────────────────────────────────────────────────
✔ Variables predictoras: 64 columnas
✔ Variable objetivo     : tiempo_resolucion_dias
────────────────────────────────────────────────────────────
📉 Outliers eliminados: 1663
📏 Escalado aplicado a: ['SLA_Minutos']
🧾 Dimensión de X escalado: (11073, 64)
🧾 Dimensión de X original: (11073, 64)
────────────────────────────────────────────────────────────
[✔] Preparación completada correctamente.
────────────────────────────────────────────────────────────

📌 Datos SIN OUTLIERS listos para modelado:
   ▸ X_ridge: (11073, 64)
   ▸ X_rf_xgb: (11073, 64)
   ▸ y: (11073,)


In [7]:
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import cross_val_score, RepeatedKFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, make_scorer
from tabulate import tabulate
import numpy as np
import pandas as pd
import time

# ─────────────────────────────────────────────────────────────
# MÉTRICAS PERSONALIZADAS
# ─────────────────────────────────────────────────────────────
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

scoring = {
    'MAE': make_scorer(mean_absolute_error),
    'RMSE': make_scorer(rmse),
    'R2': make_scorer(r2_score)
}

# ─────────────────────────────────────────────────────────────
# CONFIGURACIÓN DE MODELOS
# ─────────────────────────────────────────────────────────────
modelos = {
    'Ridge': {
        'model': Ridge(alpha=1.0, random_state=42),
        'requires_scaling': True
    },
    'RandomForest': {
        'model': RandomForestRegressor(
            n_estimators=150,
            max_depth=10,
            min_samples_leaf=4,
            random_state=42,
            n_jobs=-1
        ),
        'requires_scaling': False
    },
    'XGBoost': {
        'model': XGBRegressor(
            n_estimators=200,
            learning_rate=0.05,
            max_depth=5,
            subsample=0.8,
            colsample_bytree=0.9,
            random_state=42,
            n_jobs=-1,
            eval_metric='rmse'
        ),
        'requires_scaling': False
    }
}

# ─────────────────────────────────────────────────────────────
# FUNCIÓN DE EVALUACIÓN
# ─────────────────────────────────────────────────────────────
def evaluar_modelos(X_ridge, X_rf_xgb, y, cv_splits=5, cv_repeats=3):
    print("="*80)
    print(" INICIO DE EVALUACIÓN DE MODELOS ".center(80, "⚡"))
    print("="*80)

    cv = RepeatedKFold(n_splits=cv_splits, n_repeats=cv_repeats, random_state=42)
    resultados = []

    for nombre, config in modelos.items():
        print("\n" + "="*60)
        print(f"🚀 Evaluando: {nombre}".center(60))
        print("="*60)

        X = X_ridge if config['requires_scaling'] else X_rf_xgb
        modelo = config['model']

        print(f"\n🔹 Tipo: {'Lineal' if config['requires_scaling'] else 'Árboles'}")
        print(f"🔹 Datos: {'Escalados' if config['requires_scaling'] else 'Originales'}")

        inicio = time.time()
        resultado = {'Modelo': nombre}

        for metrica, scorer in scoring.items():
            scores = cross_val_score(modelo, X, y, scoring=scorer, cv=cv, n_jobs=-1)
            resultado[f'{metrica}_Promedio'] = np.mean(scores)
            resultado[f'{metrica}_Std'] = np.std(scores)
            resultado[f'{metrica}_Mejor'] = np.min(scores) if metrica != 'R2' else np.max(scores)
            resultado[f'{metrica}_Peor'] = np.max(scores) if metrica != 'R2' else np.min(scores)

            print(f"\n📌 {metrica}:")
            print(f"   - Promedio: {np.mean(scores):.4f}")
            print(f"   - Desviación: ±{np.std(scores):.4f}")
            print(f"   - Rango: [{np.min(scores):.4f}, {np.max(scores):.4f}]")

        resultado['Tiempo_Ejecucion'] = time.time() - inicio
        print(f"\n⏱️ Tiempo total: {resultado['Tiempo_Ejecucion']:.2f} segundos")

        resultados.append(resultado)

    df_resultados = pd.DataFrame(resultados).sort_values(by="RMSE_Promedio")
    return df_resultados

# ─────────────────────────────────────────────────────────────
# VISUALIZACIÓN DE RESULTADOS
# ─────────────────────────────────────────────────────────────
def mostrar_resultados(df):
    print("\n" + "="*80)
    print(" RESUMEN COMPARATIVO ".center(80, "📊"))
    print("="*80)

    res = df[[
        'Modelo', 
        'MAE_Promedio', 'MAE_Std',
        'RMSE_Promedio', 'RMSE_Std',
        'R2_Promedio', 'R2_Std',
        'Tiempo_Ejecucion'
    ]].copy()

    res.columns = [
        'Modelo', 
        'MAE (μ)', 'MAE (σ)',
        'RMSE (μ)', 'RMSE (σ)',
        'R² (μ)', 'R² (σ)',
        'Tiempo (s)'
    ]

    print(tabulate(
        res.round(4),
        headers='keys',
        tablefmt='pretty',
        showindex=False
    ))

    mejor = df.iloc[0]
    print(f"\n🏆 Mejor modelo: {mejor['Modelo']}")
    print(f"   - RMSE: {mejor['RMSE_Promedio']:.4f} ± {mejor['RMSE_Std']:.4f}")
    print(f"   - Tiempo ejecución: {mejor['Tiempo_Ejecucion']:.2f}s")

    if len(df) > 1:
        segundo = df.iloc[1]
        mejora = 100 * (segundo['RMSE_Promedio'] - mejor['RMSE_Promedio']) / segundo['RMSE_Promedio']
        print(f"\n🔍 Comparación con segundo mejor ({segundo['Modelo']}):")
        print(f"   - Mejora en RMSE: {mejora:.2f}%")
        print(f"   - Diferencia en tiempo: {mejor['Tiempo_Ejecucion'] - segundo['Tiempo_Ejecucion']:.2f}s")




# ─────────────────────────────────────────────────────────────
# EJECUCIÓN
# ─────────────────────────────────────────────────────────────

if __name__ == "__main__":
    # Ejemplo de uso (descomentar y reemplazar con tus datos)
    """
    # 1. Preparar datos (ejecutar primero el script de preparación)
    from preparacion_datos import preparar_datos
    X_ridge, X_rf_xgb, y, _ = preparar_datos(df, verbose=False)
    
    # 2. Evaluar modelos
    resultados = evaluar_modelos(X_ridge, X_rf_xgb, y)
    
    # 3. Mostrar resultados
    mostrar_resultados(resultados)
    """
    
    print("\n" + "="*80)
    print(" INSTRUCCIONES DE USO ".center(80, "✨"))
    print("="*80)
    print("""
1. Prepara tus datos con el script de preparación
2. Importa los DataFrames resultantes (X_ridge, X_rf_xgb, y)
3. Llama a la función evaluar_modelos()
4. Visualiza los resultados con mostrar_resultados()

Ejemplo completo:
─────────────────────────────────────────────────────
from preparacion_datos import preparar_datos

# Preparar datos
X_ridge, X_rf_xgb, y, _ = preparar_datos(df, verbose=False)

# Evaluar modelos
resultados = evaluar_modelos(X_ridge, X_rf_xgb, y)

# Mostrar resultados
mostrar_resultados(resultados)
─────────────────────────────────────────────────────
""")
    print("="*80)

StatementMeta(, a20f22a8-18c0-4f68-a6d1-70e9c1c369bf, 9, Finished, Available, Finished)


✨✨✨✨✨✨✨✨✨✨✨✨✨✨✨✨✨✨✨✨✨✨✨✨✨✨✨✨✨ INSTRUCCIONES DE USO ✨✨✨✨✨✨✨✨✨✨✨✨✨✨✨✨✨✨✨✨✨✨✨✨✨✨✨✨✨

1. Prepara tus datos con el script de preparación
2. Importa los DataFrames resultantes (X_ridge, X_rf_xgb, y)
3. Llama a la función evaluar_modelos()
4. Visualiza los resultados con mostrar_resultados()

Ejemplo completo:
─────────────────────────────────────────────────────
from preparacion_datos import preparar_datos

# Preparar datos
X_ridge, X_rf_xgb, y, _ = preparar_datos(df, verbose=False)

# Evaluar modelos
resultados = evaluar_modelos(X_ridge, X_rf_xgb, y)

# Mostrar resultados
mostrar_resultados(resultados)
─────────────────────────────────────────────────────



In [8]:
# ─────────────────────────────────────────────────────────────
# Evaluación de Modelos:
# ─────────────────────────────────────────────────────────────

# Evaluar modelos con outliers
resultados_con = evaluar_modelos(X_ridge_con, X_rf_con, y_con)
mostrar_resultados(resultados_con)

# Evaluar modelos sin outliers
resultados_sin = evaluar_modelos(X_ridge_sin, X_rf_sin, y_sin)
mostrar_resultados(resultados_sin)


StatementMeta(, a20f22a8-18c0-4f68-a6d1-70e9c1c369bf, 10, Finished, Available, Finished)

⚡⚡⚡⚡⚡⚡⚡⚡⚡⚡⚡⚡⚡⚡⚡⚡⚡⚡⚡⚡⚡⚡⚡ INICIO DE EVALUACIÓN DE MODELOS ⚡⚡⚡⚡⚡⚡⚡⚡⚡⚡⚡⚡⚡⚡⚡⚡⚡⚡⚡⚡⚡⚡⚡⚡

                     🚀 Evaluando: Ridge                     

🔹 Tipo: Lineal
🔹 Datos: Escalados

📌 MAE:
   - Promedio: 29.4728
   - Desviación: ±0.8862
   - Rango: [27.7117, 31.2230]

📌 RMSE:
   - Promedio: 54.9284
   - Desviación: ±2.6738
   - Rango: [50.7500, 59.8551]

📌 R2:
   - Promedio: 0.1642
   - Desviación: ±0.0235
   - Rango: [0.1247, 0.2089]

⏱️ Tiempo total: 4.53 segundos

                 🚀 Evaluando: RandomForest                  

🔹 Tipo: Árboles
🔹 Datos: Originales

📌 MAE:
   - Promedio: 24.7779
   - Desviación: ±0.8163
   - Rango: [23.1273, 26.2959]

📌 RMSE:
   - Promedio: 49.7297
   - Desviación: ±2.5541
   - Rango: [45.7419, 53.8463]

📌 R2:
   - Promedio: 0.3148
   - Desviación: ±0.0294
   - Rango: [0.2619, 0.3749]

⏱️ Tiempo total: 18.86 segundos

                    🚀 Evaluando: XGBoost                    

🔹 Tipo: Árboles
🔹 Datos: Originales

📌 MAE:
   - Promedio: 24.7455
   - Desviac

In [9]:
# ─────────────────────────────────────────────────────────────
# Modelos Con Outiers
# ─────────────────────────────────────────────────────────────
from xgboost import XGBRegressor
import joblib

modelo_final_xgb_con = XGBRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.9,
    random_state=42,
    n_jobs=-1,
    eval_metric='rmse'
)
modelo_final_xgb_con.fit(X_rf_con, y_con)

joblib.dump(modelo_final_xgb_con, "/lakehouse/default/Files/modelo_final_xgb_con.pkl")
print("✅ Modelo final CON OUTLIERS entrenado y guardado correctamente.")


StatementMeta(, a20f22a8-18c0-4f68-a6d1-70e9c1c369bf, 11, Finished, Available, Finished)

✅ Modelo final CON OUTLIERS entrenado y guardado correctamente.


In [10]:
# ─────────────────────────────────────────────────────────────
# Modelos Sin Outiers
# ─────────────────────────────────────────────────────────────
from xgboost import XGBRegressor
import joblib

modelo_final_xgb_sin = XGBRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.9,
    random_state=42,
    n_jobs=-1,
    eval_metric='rmse'
)
modelo_final_xgb_sin.fit(X_rf_sin, y_sin)

joblib.dump(modelo_final_xgb_sin, "/lakehouse/default/Files/modelo_final_xgb_sin.pkl")
print("✅ Modelo final SIN OUTLIERS entrenado y guardado correctamente.")



StatementMeta(, a20f22a8-18c0-4f68-a6d1-70e9c1c369bf, 12, Finished, Available, Finished)

✅ Modelo final SIN OUTLIERS entrenado y guardado correctamente.


In [11]:
# ─────────────────────────────────────────────────────────────
# Generar Predicciones (2 modelos) y Enriquecer Dashboard
# ─────────────────────────────────────────────────────────────

import joblib

# 1. Cargar ambos modelos desde el Lakehouse
modelo_con = joblib.load("/lakehouse/default/Files/modelo_final_xgb_con.pkl")
modelo_sin = joblib.load("/lakehouse/default/Files/modelo_final_xgb_sin.pkl")
print("📦 Modelos cargados correctamente (CON y SIN outliers).")

# 2. Leer dataset de modelado (ya codificado, incluye 'Caso')
df_modelo = spark.read.table("Gold.bd_aranda_gold_modelo").toPandas()
X_modelo = df_modelo.drop(columns=["Caso", "tiempo_resolucion_dias"], errors="ignore")

# 3. Generar ambas predicciones
df_modelo["tiempo_resolucion_estimado_ConOutliers"] = modelo_con.predict(X_modelo)
df_modelo["tiempo_resolucion_estimado_SinOutliers"] = modelo_sin.predict(X_modelo)

# 4. Extraer predicciones por 'Caso'
df_predicciones = df_modelo[[
    "Caso", 
    "tiempo_resolucion_estimado_ConOutliers", 
    "tiempo_resolucion_estimado_SinOutliers"
]].copy()

# 5. Leer el dashboard original desde Gold
df_dashboard = spark.read.table("Gold.bd_aranda_gold_dashboard").toPandas()

# 6. Hacer merge por 'Caso' para anexar ambas predicciones
df_dashboard_enriquecido = df_dashboard.merge(df_predicciones, on="Caso", how="left")

# 7. Limpieza de nulos e infinitos
df_dashboard_enriquecido.replace([np.inf, -np.inf], np.nan, inplace=True)
df_dashboard_enriquecido = df_dashboard_enriquecido.where(pd.notnull(df_dashboard_enriquecido), None)

# 8. Convertir a Spark DataFrame y renombrar columnas si es necesario
df_spark_resultado = spark.createDataFrame(df_dashboard_enriquecido)
df_spark_resultado_clean = df_spark_resultado.toDF(*[col.replace(" ", "_") for col in df_spark_resultado.columns])

# 9. Guardar tabla final en Gold (puedes usar otro nombre si deseas)
nombre_tabla = "Gold.bd_aranda_gold_dashboard_prediccion"
spark.sql(f"DROP TABLE IF EXISTS {nombre_tabla}")
df_spark_resultado_clean.write.mode("overwrite").saveAsTable(nombre_tabla)

print("✅ Ambas predicciones agregadas correctamente al dashboard enriquecido.")


StatementMeta(, a20f22a8-18c0-4f68-a6d1-70e9c1c369bf, 13, Finished, Available, Finished)

📦 Modelos cargados correctamente (CON y SIN outliers).
✅ Ambas predicciones agregadas correctamente al dashboard enriquecido.


In [13]:
#%%sql
#SELECT * from bd_aranda_gold_dashboard_prediccion

StatementMeta(, a20f22a8-18c0-4f68-a6d1-70e9c1c369bf, 15, Finished, Available, Finished)

<Spark SQL result set with 1000 rows and 68 fields>